# Bronze Payload Deduplication

Identifies duplicate payloads in append-only Bronze tables.

**Purpose**: Mark duplicates without deleting data  
**Pattern**: Append-only integrity checks  
**No deletion**: Maintains immutable Bronze principle

In [0]:
# Deduplication parameters
dbutils.widgets.text("source_table", "workspace.bronze.bronze_job_snapshot", "Source Table (catalog.schema.table)")
dbutils.widgets.text("dedupe_keys", "payload_hash", "Dedupe Key Columns (comma-separated)")
dbutils.widgets.text("batch_id", "", "Batch ID")
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "bronze", "Schema")

from uuid import uuid4

# Get values with fallbacks (handles empty strings from notebook.run arguments)
source_table = dbutils.widgets.get("source_table") or "workspace.bronze.bronze_job_snapshot"
dedupe_keys_str = dbutils.widgets.get("dedupe_keys") or "payload_hash"
dedupe_keys = [k.strip() for k in dedupe_keys_str.split(",") if k.strip()]
catalog = dbutils.widgets.get("catalog") or "workspace"
schema = dbutils.widgets.get("schema") or "bronze"

# batch_id should be the INGESTION batch_id to process
# If not provided, get the latest ingestion batch from source table
batch_id_param = dbutils.widgets.get("batch_id")
if batch_id_param:
    batch_id = batch_id_param
    print(f"Processing specific batch: {batch_id}")
else:
    # Get latest ingestion batch_id from source table
    latest_batch = spark.sql(f"""
        SELECT batch_id 
        FROM {source_table}
        WHERE batch_id IS NOT NULL
        ORDER BY ingestion_timestamp DESC 
        LIMIT 1
    """).collect()
    
    if latest_batch:
        batch_id = latest_batch[0]["batch_id"]
        print(f"📌 Auto-detected latest ingestion batch_id: {batch_id}")
    else:
        raise ValueError(f"No batch_id found in {source_table}")

print(f"Source table: {source_table}")
print(f"Dedupe keys: {dedupe_keys}")
print(f"Batch ID: {batch_id}")

In [0]:
# Create table to track duplicates using Python f-string for proper variable substitution
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.dedupe_tracking (
  dedupe_id STRING,
  source_table STRING,
  batch_id STRING,
  dedupe_key_hash STRING,
  first_seen_record_id STRING,
  first_seen_batch_id STRING,
  duplicate_count INT,
  first_seen_timestamp TIMESTAMP,
  last_seen_timestamp TIMESTAMP,
  tracking_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Tracks duplicate payload occurrences in Bronze tables'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
)
""")

print(f"Ensured {catalog}.{schema}.dedupe_tracking table exists")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime
import hashlib

# Read source table
df = spark.table(source_table)

# Create hash of dedupe keys
concat_expr = F.concat_ws("|", *[F.coalesce(F.col(k).cast("string"), F.lit("NULL")) for k in dedupe_keys])
df_with_hash = df.withColumn("dedupe_key_hash", F.sha2(concat_expr, 256))

# Identify duplicates (keep first occurrence)
window_spec = Window.partitionBy("dedupe_key_hash").orderBy(F.col("ingestion_timestamp").asc())
df_ranked = df_with_hash.withColumn("row_num", F.row_number().over(window_spec))

# Aggregate duplicate info
dupe_stats = df_ranked.groupBy("dedupe_key_hash").agg(
    F.count("*").cast("int").alias("duplicate_count"),
    F.min(F.when(F.col("row_num") == 1, F.col("snapshot_id"))).alias("first_seen_record_id"),
    F.min(F.when(F.col("row_num") == 1, F.col("batch_id"))).alias("first_seen_batch_id"),
    F.min("ingestion_timestamp").alias("first_seen_timestamp"),
    F.max("ingestion_timestamp").alias("last_seen_timestamp")
).filter(F.col("duplicate_count") > 1)

# Add metadata
from uuid import uuid4

tracking_df = dupe_stats.select(
    F.lit(str(uuid4())).alias("dedupe_id"),
    F.lit(source_table).alias("source_table"),
    F.lit(batch_id).alias("batch_id"),
    F.col("dedupe_key_hash"),
    F.col("first_seen_record_id"),
    F.col("first_seen_batch_id"),
    F.col("duplicate_count"),
    F.col("first_seen_timestamp"),
    F.col("last_seen_timestamp"),
    F.lit(datetime.now()).alias("tracking_timestamp")
)

dupes_found = tracking_df.count()
print(f"Found {dupes_found} duplicate key(s) in {source_table}")

In [0]:
# Write to tracking table
if dupes_found > 0:
    target_table = f"{catalog}.{schema}.dedupe_tracking"
    tracking_df.write.mode("append").saveAsTable(target_table)
    print(f"Dedupe tracking written to {target_table}")
    
    # Show summary
    print("\nDuplicate Summary:")
    tracking_df.select(
        "dedupe_key_hash",
        "duplicate_count",
        "first_seen_timestamp",
        "last_seen_timestamp"
    ).show(20, truncate=False)
else:
    print("No duplicates found")